# Plumb: fine-tune Laya (Kaggle, GPU T4 ×2)

Fine-tunes the English Laya checkpoint (`convaiinnovations/laya`, 421M) on Plumb's writing signals, the "which word" pointer and mistake types. One run trains **two data mixes for 4 epochs each**; every epoch from 2 on is a **candidate**. Each candidate gets the same accuracy check as the app plus the **ship bar** (at least as good as the live model on real learner writing, pointer and types right ≥ 90%). Only the best passing candidate is packaged into `plumb-model.zip`; if none passes, nothing is packaged.

**Before you run (one time):** Settings → Accelerator → **GPU T4 ×2**, Internet on; add the private dataset holding `train.jsonl`, `catalogue.json`, `engine.zip` and `ship_cases.json`. Then *Save Version → Save & Run All*. Expect about 4–5 hours.


In [ ]:
LABEL_SMOOTHING = 0.05  # keeps confidence from saturating on one-hot labels

In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Please switch your Kaggle Accelerator: on the right sidebar, go to Notebook options -> "
    "Accelerator -> select GPU T4 x2."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")


In [ ]:
!pip install -q "laya==0.3.20" "datasets>=3.0.0" safetensors huggingface_hub
import glob, os, zipfile, json
inputs = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert inputs, "Upload train.jsonl, catalogue.json and engine.zip as a Kaggle dataset first (see step 2)."
DATA_DIR = os.path.dirname(inputs[0])
import shutil
# Kaggle may unpack uploaded zips itself, so accept either the zip or the unpacked folder.
zips = glob.glob("/kaggle/input/**/engine.zip", recursive=True)
if zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/kaggle/working")
else:
    pkg = glob.glob("/kaggle/input/**/writing_signals/__init__.py", recursive=True)
    assert pkg, "engine.zip (or its unpacked engine folder) is missing from the dataset."
    shutil.copytree(os.path.dirname(os.path.dirname(pkg[0])), "/kaggle/working/engine", dirs_exist_ok=True)
print("Data:", DATA_DIR, "| engine unpacked to /kaggle/working/engine")

## 1. Turn each labelled sentence into a Laya training item

In [ ]:
import json, torch
from collections import Counter
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES
import sys
sys.path.insert(0, "/kaggle/working/engine")
# The "which word is wrong?" question is built per sentence, by the same function the app uses.
from writing_signals.catalogue import MISTAKE_TYPE_QUESTION, locate_question, type_state

model_dir = snapshot_download("convaiinnovations/laya")
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
cfg = json.load(open(os.path.join(model_dir, "rl_agent_config.json")))
catalogue = json.load(open(os.path.join(DATA_DIR, "catalogue.json")))
questions = catalogue["questions"]
# Flow rows are sentence pairs, shown to the model exactly as the app shows them.
flow_question, flow_format = catalogue.get("flow_question"), catalogue.get("flow_state_format")

def options(q):
    # Target order Laya expects: noul is [false, true]; choice follows criteria keys; score follows levels.
    return ["no", "yes"] if q["type"] == "noul" else list(q["criteria"])

items, skipped = [], 0
rows = [json.loads(l) for l in open(os.path.join(DATA_DIR, "train.jsonl"))]
for r in rows:
    flow = r["signal"] == "flow"
    kind = r["signal"] == "mistake_type"  # what kind of mistake the marked word is
    q = (flow_question if flow else locate_question(r["sentence"]) if r["signal"] == "locate"
         else MISTAKE_TYPE_QUESTION if kind else questions[r["signal"]])
    state = (flow_format.format(previous=r["previous"], sentence=r["sentence"]) if flow
             else type_state(r["sentence"], r["word"]) if kind else r["sentence"])
    labels = options(q)
    k = len(labels)
    target = [LABEL_SMOOTHING / (k - 1)] * k
    target[labels.index(r["expected"])] = 1 - LABEL_SMOOTHING
    crit = q.get("criteria", {})
    seq, markers = build_sequence(tok, state, {"t": q["type"], "ins": q["instructions"], "crit": crit},
                                  512, cfg["head_max_len"])
    if len(markers) != len(render_options({"t": q["type"], "crit": crit})):
        skipped += 1
        continue
    items.append({"signal": r["signal"], "ids": seq, "markers": markers, "qtype": QTYPES[q["type"]],
                  "target": target, "label": labels.index(r["expected"])})

# Mix A: everything. Mix B: the same with every other pointer and type row left out, so grammar
# carries more weight. Both are candidates; the ship bar decides.
torch.save(items, "/kaggle/working/items_A.pt")
items_B, seen = [], 0
for it in items:
    if it["signal"] in ("locate", "mistake_type"):
        seen += 1
        if seen % 2 == 0:
            continue
    items_B.append(it)
torch.save(items_B, "/kaggle/working/items_B.pt")
print(f"mix A {len(items)} items, mix B {len(items_B)} items")
print(f"{len(items)} training items ({skipped} skipped)")
print(Counter(r["signal"] for r in rows))

## 2. Accuracy check on the base model (for comparison)

In [ ]:
%cd /kaggle/working/engine
!python -m writing_signals.eval --sample 0 --device cuda --threads 2 2>&1 | grep -v -i warn
%cd /kaggle/working

## 3. Fine-tune (Laya's RLCD recipe, both T4s)

In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def save_candidate(model, out_dir, calib_items, cfg, tok, device):
    """Fits calibration temperatures on the held-out slice and saves a complete model folder."""
    model.eval()
    calib_preds = []
    with torch.no_grad():
        for c_idx in range(0, len(calib_items), 16):
            c_chunk = calib_items[c_idx:c_idx + 16]
            cb = collate_train_batch(c_chunk, tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                l_sub, _ = model(cb["input_ids"].to(device), cb["attention_mask"].to(device), cb["marker_pos"].to(device),
                                 cb["marker_mask"].to(device), cb["qtype"].to(device))
            l_np = l_sub.float().cpu().numpy()
            for r, it in enumerate(c_chunk):
                calib_preds.append((it["qtype"], l_np[r, :len(it["markers"])], it["target"]))
    fitted_temps = [1.2, 1.2, 1.2]
    try:
        for qt in range(3):
            sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
            if sel:
                fitted_temps[qt] = fit_one_temp(sel)
    except Exception as e:
        print("Temperature fitting fallback:", e)
    os.makedirs(out_dir, exist_ok=True)
    save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}, os.path.join(out_dir, "model.safetensors"))
    model.encoder.config.save_pretrained(os.path.join(out_dir, "encoder"))
    tok.save_pretrained(os.path.join(out_dir, "tokenizer"))
    saved = {**cfg, "fine_tuned": True, "model_name": "laya-writing-signals", "temperature": fitted_temps}
    saved.pop("temperature_by_options", None)  # this fit is per type; inherited overrides would hide it
    with open(os.path.join(out_dir, "rl_agent_config.json"), "w") as f:
        json.dump(saved, f, indent=2)
    print(f"  Candidate saved: {out_dir} (temperatures {[round(t, 3) for t in fitted_temps]})")


def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    items_path = sys.argv[3]
    EPOCHS = int(sys.argv[4])
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 512  # single sentences are short
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    
    all_items = torch.load(items_path, weights_only=False)

    # Hold the calibration slice out of training before sharding. Temperatures fitted on
    # items the run has already trained on measure the fit rather than the calibration: the
    # model is near-certain and near-correct on them, so the optimiser has nothing to soften
    # and returns a degenerate scale. The seed is fixed and rank-independent, so every rank
    # withholds exactly the same items and none of them reaches a training batch.
    CALIB_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922).shuffle(order)
    n_calib = min(CALIB_MAX, len(all_items) // 10)
    calib_items = [all_items[i] for i in sorted(order[:n_calib])]
    train_items = [all_items[i] for i in sorted(order[n_calib:])]
    my_items = train_items[rank::world_size]
    
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 4       # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    GROUP_SIZE = 4       # GRPO baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            
            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            # 1. Sample G noisy logit distributions with zero-mean projection
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            # 2. Evaluate proper scoring reward (w_sph=0.75 for soft target matching)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            # 3. Policy gradient loss + full 1.0 soft cross-entropy guidance
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            
            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Epochs 2 and later are candidates: calibrated and saved as their own model, so the notebook
        # can score every one against the ship bar and keep only the best.
        if rank == 0 and epoch + 1 >= 2:
            save_candidate(model, os.path.join(output_dir, f"e{epoch + 1}"), calib_items, cfg, tok, device)
            model.train()
        dist.barrier()

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


In [ ]:
EPOCHS = 4
for mix in ("A", "B"):
    out = f"/kaggle/working/cand_{mix}"
    !torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py {model_dir} {out} /kaggle/working/items_{mix}.pt {EPOCHS}


## 4. Score every candidate against the ship bar

Each candidate: the app's accuracy check (every signal must pass its bar), then the report's cases and held-out pointer/type cases through `writing_signals.eval.ship`: the same scorer the local check uses.


In [ ]:
import glob, json, os, subprocess, sys, torch
sys.path.insert(0, "/kaggle/working/engine")
from writing_signals.engine import SignalEngine
from writing_signals.eval import ship

# The live model's numbers (run 3) on the same cases: a candidate must be at least this good.
FLOORS = {"fce_caught": 0.7843, "jfleg_caught": 0.8844, "fce_false_alarms": 0.0737, "blimp_false_alarms": 0.1851}
ship_data = json.load(open(glob.glob("/kaggle/input/**/ship_cases.json", recursive=True)[0]))
candidates = {}
for cand in sorted(glob.glob("/kaggle/working/cand_*/e*")):
    name = "/".join(cand.split("/")[-2:])
    before = set(glob.glob("/kaggle/working/engine/data/reports/accuracy-*.json"))
    subprocess.run(["python", "-m", "writing_signals.eval", "--sample", "0", "--device", "cuda", "--threads", "2",
                    "--checkpoint", cand], cwd="/kaggle/working/engine", capture_output=True)
    report = sorted(set(glob.glob("/kaggle/working/engine/data/reports/accuracy-*.json")) - before)[-1]
    signals = json.load(open(report))["signals"]
    engine = SignalEngine(checkpoint=cand, device="cuda")
    engine.knows_types = True  # every candidate here was trained on mistake types
    m = ship.evaluate(engine, ship_data)
    m["signals_pass"] = all(s["pass"] for s in signals.values())
    m["report"] = report
    candidates[name] = m
    del engine; torch.cuda.empty_cache()
    print(name, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items() if k != "report"}, "PASS" if ship.passes(m, FLOORS) else "fails")

WINNER = ship.pick(candidates, FLOORS)
json.dump({"floors": FLOORS, "candidates": candidates, "winner": WINNER}, open("/kaggle/working/ship_report.json", "w"), indent=1)
print("\nWinner:", WINNER or "none passed the ship bar: nothing will be packaged")


## 5. Package the winner

`plumb-model.zip` holds the winning candidate, its accuracy report and `ship_report.json` (every candidate's numbers). Nothing is packaged if no candidate passed.


In [ ]:
import shutil
if WINNER:
    win = glob.glob(f"/kaggle/working/{WINNER}")[0]
    os.makedirs(os.path.join(win, "eval"), exist_ok=True)
    rep = candidates[WINNER]["report"]
    for ext in (".json", ".md"):
        shutil.copy(rep[:-5] + ext, os.path.join(win, "eval"))
    shutil.copy("/kaggle/working/ship_report.json", os.path.join(win, "eval"))
    json.dump({"catalogue": catalogue["version"], "run": 5, "candidate": WINNER, "thresholds": {
        "pointer": candidates[WINNER]["pointer_threshold"], "type": candidates[WINNER]["type_threshold"]}},
        open(os.path.join(win, "plumb_model.json"), "w"))
    archive = shutil.make_archive("/kaggle/working/plumb-model", "zip", win)
    print(f"Ready to download: {archive} ({os.path.getsize(archive) / 1e6:.0f} MB), winner {WINNER}")
else:
    print("No candidate passed: nothing packaged. The live model stays.")
# Candidates are large; keep only the zip and the report in the output.
for d in glob.glob("/kaggle/working/cand_*"):
    shutil.rmtree(d, ignore_errors=True)
